# Probar Walmart Visión en Google Colab

Usa los modelos entrenados V1 y V2 base para analizar una foto, sin volver a entrenar. Ejecuta las celdas en orden.

Opcional: selecciona una GPU en **Entorno de ejecución → Cambiar tipo de entorno de ejecución**. Si no hay GPU, se usará CPU.

**Los archivos que selecciones se subirán al entorno de Google Colab.** No se monta Drive ni se publica una web. El ZIP incluye los modelos y el código; no necesitas subir el dataset.


In [ ]:
%pip -q install ultralytics==8.4.174 pillow==12.3.0 "ipywidgets>=8,<9"


## Cargar el paquete

Selecciona el ZIP completo **Walmart_Vision_Cuatro_Vertices.zip** cuando aparezca el botón. La carga puede tardar según tu conexión.


In [ ]:
from google.colab import files
from pathlib import Path
from io import BytesIO
import sys, zipfile

paquetes = files.upload()
zip_names = [name for name in paquetes if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError('Selecciona un único ZIP de Walmart Visión.')
destino = Path('/content/walmart_vision').resolve()
destino.mkdir(exist_ok=True)
with zipfile.ZipFile(BytesIO(paquetes[zip_names[0]])) as archive:
    if sum(item.file_size for item in archive.infolist()) > 100 * 1024 * 1024:
        raise ValueError('El ZIP supera el tamaño esperado para este paquete.')
    for item in archive.infolist():
        if not (destino / item.filename).resolve().is_relative_to(destino):
            raise ValueError('El ZIP contiene una ruta inválida.')
    archive.extractall(destino)
carpeta = destino / 'Walmart_Vision'
if not (carpeta / 'modelos.json').is_file():
    raise ValueError('El archivo no corresponde al ZIP de Walmart Visión.')
sys.path.insert(0, str(carpeta))
from inferencia import ModelRunner
visor = ModelRunner(carpeta)
print('Paquete listo. Modelos:', ', '.join(m['id'] for m in visor.available()))


## 0. Seleccionar versión y modelo

Selecciona el modelo antes de subir la foto. Para comparar, puedes repetir el análisis con el otro modelo sobre la misma imagen.


In [ ]:
import ipywidgets as widgets
from IPython.display import display
modelos = visor.available()
versiones = [v for v in ('V2', 'V1') if any(('V2' if m['id'].startswith('V2-') else 'V1') == v for m in modelos)]
version = widgets.Dropdown(options=versiones, description='Versión:')
selector = widgets.Dropdown(description='Modelo:')
def actualizar_modelos(change=None):
    selector.options = [m['id'] for m in modelos if ('V2' if m['id'].startswith('V2-') else 'V1') == version.value]
    selector.value = next((m for m in selector.options if '26' in m), selector.options[0])
version.observe(actualizar_modelos, names='value')
actualizar_modelos()
display(version, selector)


## Subir una foto

Elige una foto JPG, PNG o WebP de hasta 15 MB y 24 megapíxeles. Las fotografías nuevas ayudan a observar el comportamiento; esta prueba no calcula precisión contra anotaciones.


In [ ]:
fotos = files.upload()
if len(fotos) != 1:
    raise ValueError('Selecciona una sola fotografía.')
nombre_foto, contenido_foto = next(iter(fotos.items()))
print('Foto lista:', nombre_foto)


## Analizar

La confianza mínima predeterminada es 0,25. Puedes cambiarla en la siguiente celda; no representa el porcentaje de producto del nivel. No evaluable no significa vacío.

Para probar el otro modelo, cambia el selector y vuelve a ejecutar esta celda. Para usar otra foto, repite la celda de carga y luego esta.


In [ ]:
confianza = 0.25
usar_bordes = True  # False para ejecutar solo YOLO.
prediccion = visor.analyze(contenido_foto, selector.value, confianza,
                          method='yolo_paulo' if usar_bordes else 'yolo')
print(f"{prediccion['model']} · {prediccion['device']} · {prediccion['seconds']} s")
print('Niveles detectados:', len(prediccion['detections']))
print('Estados YOLO:', prediccion['counts'])
if usar_bordes:
    print('Ocupación estimada: rojo bajo el 30 %; verde desde el 30 %. Estimación experimental.')
    for nivel in prediccion['detections']:
        lectura = nivel['occupancy_estimate']
        porcentaje = str(lectura['pct']) + ' %' if lectura['pct'] is not None else 'Sin medición'
        print('Nivel', nivel['id'], porcentaje, lectura['label'])
from inferencia import read_photo
original = read_photo(contenido_foto)
original_view, niveles_view, bordes_view, comparar_view = [widgets.Output() for _ in range(4)]
with original_view:
    display(original)
with niveles_view:
    display(prediccion['image'])
with bordes_view:
    if usar_bordes:
        print('Amarillo: señal experimental. Los colores de Niveles usan la ocupación estimada y el umbral del 30 %.')
        display(prediccion['occupancy_image'])
        display(prediccion['filtered_edges_image'])
        display(prediccion['edges_image'])
    else:
        print('Activa usar_bordes y ejecuta de nuevo para ver estos mapas.')
with comparar_view:
    print('Original')
    display(original)
    print('Niveles detectados por', prediccion['model'])
    display(prediccion['image'])
vistas = widgets.Tab(children=[original_view, niveles_view, bordes_view, comparar_view])
for i, titulo in enumerate(('Original', 'Niveles', 'Bordes', 'Comparar')):
    vistas.set_title(i, titulo)
vistas.selected_index = 1
display(vistas)
ruta_resultado = Path('/content') / f"resultado_{prediccion['model']}.jpg"
prediccion['image'].save(ruta_resultado, quality=92)


## Descargar el resultado


In [ ]:
files.download(str(ruta_resultado))
